# Pengantar Convnet: MNIST dengan Konvolusi

**Buku:** *Deep Learning dengan Python* — Bab 5, Deep Learning untuk Visi Komputer

Convnet sederhana (Conv2D + MaxPooling2D) mengalahkan jaringan Dense dari Bab 2 pada MNIST.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- Model dibangun dengan API fungsional Keras 3 (`keras.Input`).
- Label bilangan bulat + `sparse_categorical_crossentropy`.
- Ditambah perbandingan model **tanpa** max-pooling (jumlah parameter meledak).

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

In [ ]:
from keras import layers

masukan = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu")(masukan)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(64, 3, activation="relu")(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(64, 3, activation="relu")(x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation="relu")(x)
keluaran = layers.Dense(10, activation="softmax")(x)
model = keras.Model(masukan, keluaran)
model.summary()

In [ ]:
from keras.datasets import mnist

(x_latih, y_latih), (x_uji, y_uji) = mnist.load_data()
x_latih = x_latih.reshape((60000, 28, 28, 1)).astype("float32") / 255
x_uji = x_uji.reshape((10000, 28, 28, 1)).astype("float32") / 255

model.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_latih, y_latih, epochs=5, batch_size=64)
loss, akurasi = model.evaluate(x_uji, y_uji, verbose=0)
print(f"Akurasi uji convnet: {akurasi:.2%}  (Dense pada Bab 2: ±97,8%)")

## Mengapa perlu downsampling?
Tanpa max-pooling, peta fitur tetap besar sehingga lapisan `Dense` di atasnya memiliki jutaan parameter, dan jendela lapisan atas hanya "melihat" area kecil citra.

In [ ]:
tanpa_pool = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation="relu"),
    layers.Conv2D(64, 3, activation="relu"),
    layers.Conv2D(64, 3, activation="relu"),
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
print(f"Parameter tanpa pooling: {tanpa_pool.count_params():,}")
print(f"Parameter dengan pooling: {model.count_params():,}")